# NB-06-03 — Model Comparison and Ensembles

**Module:** M06 · **Time:** 60–90 minutes · **Prerequisites:** L06-05 to L06-08

Run seven model families on the same split, then on the same folds, and decide which one earns its
complexity. The comparison discipline matters more than the winner.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M06-machine-learning-fundamentals" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

code directory: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M06-machine-learning-fundamentals\code


In [2]:
import numpy as np
import datasets as ds
import evaluation as ev
from prep import Pipeline, StandardScaler, SimpleImputer, train_test_split
from classifiers import (LogisticRegressionGD, KNNClassifier, GaussianNB,
                         DecisionTreeClassifier, RandomForestClassifier,
                         GradientBoostingClassifier, LinearSVM)

## 2. One split, many models

In [3]:
frame = ds.loans()
cols = ["annual_income_k", "loan_amount_k", "credit_score",
        "employment_years", "debt_ratio", "default_history"]
X = frame[cols].fillna(frame[cols].median()).to_numpy()
y = frame["approved"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, 0.25, seed=0, stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr, Xte = scaler.transform(Xtr), scaler.transform(Xte)

models = {
    "logistic": LogisticRegressionGD(lr=0.5, epochs=800),
    "kNN 15": KNNClassifier(k=15),
    "GaussianNB": GaussianNB(),
    "tree d5": DecisionTreeClassifier(max_depth=5, min_samples_leaf=5),
    "tree unlimited": DecisionTreeClassifier(),
    "forest 100": RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5),
    "boosting 100": GradientBoostingClassifier(n_estimators=100, max_depth=2),
    "linear SVM": LinearSVM(C=0.5, lr=0.02, epochs=400),
}
for name, m in models.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xte)
    print(f"{name:15s} train {m.score(Xtr, ytr):.3f} test {m.score(Xte, yte):.3f} "
          f"AUC {ev.roc_auc_score(yte, p):.3f}")

logistic        train 0.784 test 0.783 AUC 0.798
kNN 15          train 0.790 test 0.743 AUC 0.766
GaussianNB      train 0.766 test 0.711 AUC 0.758
tree d5         train 0.801 test 0.717 AUC 0.696
tree unlimited  train 1.000 test 0.679 AUC 0.597


forest 100      train 0.834 test 0.762 AUC 0.802


boosting 100    train 0.825 test 0.762 AUC 0.790
linear SVM      train 0.775 test 0.778 AUC 0.795


The unlimited tree's 1.000 training accuracy and lowest AUC is the lesson in one row:
memorisation is not generalisation.

## 3. Same folds, mean and spread

In [4]:
def make_log():
    return Pipeline([("impute", SimpleImputer("median")), ("scale", StandardScaler()),
                     ("model", LogisticRegressionGD(lr=0.5, epochs=400))])

def make_forest():
    return RandomForestClassifier(n_estimators=40, max_depth=6, min_samples_leaf=5)

for name, factory in [("logistic", make_log), ("forest 40", make_forest)]:
    scores = ev.cross_val_score(factory, X, y, k=5, stratified=True)
    print(f"{name:10s} {np.round(scores, 4)} mean {scores.mean():.4f} std {scores.std():.4f}")

logistic   [0.7933 0.7567 0.8133 0.7833 0.78  ] mean 0.7853 std 0.0185


forest 40  [0.7633 0.7667 0.8067 0.77   0.7633] mean 0.7740 std 0.0165


## 4. Learning curves: variance or bias?

In [5]:
for label, factory in [
    ("unlimited", lambda: DecisionTreeClassifier()),
    ("depth 4", lambda: DecisionTreeClassifier(max_depth=4, min_samples_leaf=10)),
]:
    curve = ev.learning_curve(factory, X, y, fractions=(0.2, 0.5, 1.0), k=3)
    print(label)
    print(curve.round(3).to_string(index=False))

unlimited
 fraction  n_train  train  validation
      0.2      200    1.0       0.647
      0.5      500    1.0       0.653
      1.0     1000    1.0       0.683
depth 4
 fraction  n_train  train  validation
      0.2      200  0.808       0.747
      0.5      500  0.795       0.743
      1.0     1000  0.775       0.734


## 5. Naive Bayes and k-NN on text-like features

In [6]:
spam = ds.emails()
scols = ["word_count", "num_links", "uppercase_words", "exclamation_marks",
         "has_attachment", "sender_known", "hour_sent", "contains_free"]
Xs, ys = spam[scols].to_numpy(), spam["is_spam"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, 0.25, seed=0, stratify=ys)
sc = StandardScaler().fit(Xtr)

for name, model in [("GaussianNB", GaussianNB()),
                    ("kNN 5", KNNClassifier(k=5)),
                    ("logistic", LogisticRegressionGD(lr=0.5, epochs=800, class_weight="balanced"))]:
    model.fit(sc.transform(Xtr), ytr)
    p = model.predict_proba(sc.transform(Xte))[:, 1]
    print(f"{name:12s} acc {model.score(sc.transform(Xte), yte):.3f} "
          f"P {ev.precision(yte, (p >= .5).astype(int)):.3f} "
          f"R {ev.recall(yte, (p >= .5).astype(int)):.3f} "
          f"AUC {ev.roc_auc_score(yte, p):.3f}")

GaussianNB   acc 0.970 P 0.855 R 0.979 AUC 0.994
kNN 5        acc 0.950 P 0.923 R 0.750 AUC 0.963
logistic     acc 0.983 P 0.939 R 0.958 AUC 0.998


## 6. Threshold and class weights change the product

In [7]:
attr = ds.attrition()
acols = ["age", "monthly_income_k", "years_at_company", "job_satisfaction", "overtime",
         "distance_from_home_km", "num_companies_worked", "environment_satisfaction", "job_level"]
Xa, ya = attr[acols].to_numpy(), attr["attrition"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xa, ya, 0.25, seed=0, stratify=ya)
asc = StandardScaler().fit(Xtr)
for weights in [None, "balanced"]:
    m = LogisticRegressionGD(lr=0.5, epochs=1000, class_weight=weights).fit(asc.transform(Xtr), ytr)
    p = m.predict_proba(asc.transform(Xte))[:, 1]
    print(weights, "acc", round(m.score(asc.transform(Xte), yte), 3),
          "P", round(ev.precision(yte, (p >= .5).astype(int)), 3),
          "R", round(ev.recall(yte, (p >= .5).astype(int)), 3))

None acc 0.843 P 0.538 R 0.125
balanced acc 0.68 P 0.306 R 0.786


## 7. Reflect

1. Which model would you choose for the loan problem if the bank needs an explanation per
   decision?
2. Which row of the comparison table would you show a stakeholder who only cares about accuracy,
   and what would you add?
3. Why do the two learning curves call for different fixes?

Next: **NB-06-04** for the unsupervised work.